# Bank Marketing — Term Deposit Subscription Prediction

**Dataset:** UCI Bank Marketing Dataset (`bank-full.csv`) — 45,211 records, 16 features  
**Goal:** Predict whether a client will subscribe to a term deposit (`y = yes/no`)  
**Approach:** Explore the data → engineer features → compare classifiers → tune the best one

---

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Make plots look a bit cleaner
sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 100

## 1. Load & Preview the Dataset

In [ ]:
df = pd.read_csv("bank-full.csv", sep=";")
print(f"Shape: {df.shape}")
df.head()

In [ ]:
df.info()

In [ ]:
print("=== Missing values ===")
print(df.isna().sum())

print("\n=== Target value counts ===")
print(df["y"].value_counts())

print("\n=== Target proportions ===")
print(df["y"].value_counts(normalize=True).round(3))

In [ ]:
df.describe()

In [ ]:
# Summarise the categorical columns
cat_summary_cols = df.select_dtypes(include="object").columns.tolist()
df[cat_summary_cols].describe()

## 2. Exploratory Data Analysis (EDA)

In [ ]:
print("Target class distribution:")
print(df["y"].value_counts())
print(f"\nClass imbalance ratio — 'no': {df['y'].value_counts(normalize=True)['no']:.1%}  'yes': {df['y'].value_counts(normalize=True)['yes']:.1%}")

In [ ]:
num_cols = df.select_dtypes(include=["int64", "float64"]).columns.tolist()
cat_cols = df.select_dtypes(include="object").columns.tolist()

print("Numeric columns :", num_cols)
print("Categorical columns:", cat_cols)

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
counts = df["y"].value_counts()
bars = ax.bar(counts.index, counts.values, color=["#4C72B0", "#DD8452"], edgecolor="black")
ax.bar_label(bars, fmt="%d")
ax.set_title("Target Distribution (y)", fontsize=14, fontweight="bold")
ax.set_xlabel("Subscribed to term deposit?")
ax.set_ylabel("Count")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 7))
corr = df[num_cols].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))  # show only lower triangle
sns.heatmap(corr, mask=mask, annot=True, fmt=".2f", cmap="coolwarm",
            linewidths=0.5, square=True)
plt.title("Correlation Heatmap — Numeric Features", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
df[num_cols].hist(figsize=(12, 8), bins=25, edgecolor="black", color="steelblue")
plt.suptitle("Distribution of Numerical Features", fontsize=15, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Subscription rate by key categorical features
df_plot = df.copy()
df_plot["target_num"] = df_plot["y"].map({"no": 0, "yes": 1})

plot_cols = ["job", "marital", "education", "housing", "loan", "contact", "poutcome"]

for col in plot_cols:
    order = (
        df_plot.groupby(col)["target_num"]
        .mean()
        .sort_values(ascending=False)
        .index
    )
    plt.figure(figsize=(9, 4))
    sns.barplot(data=df_plot, x=col, y="target_num", order=order, color="steelblue")
    plt.title(f"Subscription Rate by {col.title()}", fontsize=13, fontweight="bold")
    plt.ylabel("Avg Subscription Rate")
    plt.xlabel(col.title())
    plt.xticks(rotation=40, ha="right")
    plt.tight_layout()
    plt.show()

## 3. Feature Engineering

We create a few extra features that might help the model:
- **had_previous_contact** — was the client contacted before this campaign?
- **previous_success** — did the last campaign end successfully?
- **has_any_loan** — does the client carry a housing or personal loan?
- **balance_group** — bucket account balance into segments
- **age_group** — bucket age into life stages
- **campaign_group** — bucket number of contacts into intensity levels

> We also **drop `duration`** at this stage.  
> Duration is measured *after* the call ends, so including it would leak future information into the model — it can't be known at prediction time.

In [ ]:
# Binary flag: was there any previous contact?
df["had_previous_contact"] = (df["previous"] > 0).astype(int)

# Binary flag: did the previous campaign succeed?
df["previous_success"] = (df["poutcome"] == "success").astype(int)

# Binary flag: does the client have any active loan?
df["has_any_loan"] = ((df["housing"] == "yes") | (df["loan"] == "yes")).astype(int)

# Bucket account balance
df["balance_group"] = pd.cut(
    df["balance"],
    bins=[-np.inf, 0, 1000, 5000, np.inf],
    labels=["negative_or_zero", "low", "medium", "high"]
)

# Bucket age into life stages
df["age_group"] = pd.cut(
    df["age"],
    bins=[17, 30, 45, 60, 100],
    labels=["young", "adult", "middle_age", "senior"]
)

# Bucket campaign contact intensity
df["campaign_group"] = pd.cut(
    df["campaign"],
    bins=[0, 1, 3, 6, np.inf],
    labels=["1_contact", "2_to_3", "4_to_6", "7_plus"]
)

print("New feature columns added:", ["had_previous_contact", "previous_success",
       "has_any_loan", "balance_group", "age_group", "campaign_group"])
df.shape

## 4. Prepare Features & Split Data

In [ ]:
# Encode target (no → 0, yes → 1)
df["y"] = df["y"].map({"no": 0, "yes": 1}).astype(int)

# Drop duration — data leakage risk (only known after the call)
df_clean = df.drop(columns="duration").copy()

# Cast categorical columns so the pipeline handles them correctly
cat_columns_final = [
    "job", "marital", "education", "default", "housing", "loan",
    "contact", "month", "poutcome", "balance_group", "age_group", "campaign_group"
]
for col in cat_columns_final:
    df_clean[col] = df_clean[col].astype("category")

X = df_clean.drop(columns="y")
y = df_clean["y"]

print(f"Features: {X.shape[1]}  |  Samples: {X.shape[0]}")
print(f"Class balance — 0: {(y==0).sum()}  |  1: {(y==1).sum()}")

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y  # stratify keeps class ratio
)

print(f"Train: {X_train.shape[0]} rows  |  Test: {X_test.shape[0]} rows")

## 5. Baseline — Dummy Classifier

Before training any real model we set a **baseline**: a dummy classifier that always predicts the majority class.  
Any model we build should beat these numbers to be worth anything.

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

dummy_clf = DummyClassifier(strategy="most_frequent")
dummy_clf.fit(X_train, y_train)
y_pred_dummy = dummy_clf.predict(X_test)

print("Baseline (majority-class) scores:")
print(f"  Accuracy : {accuracy_score(y_test, y_pred_dummy):.4f}")
print(f"  Precision: {precision_score(y_test, y_pred_dummy, zero_division=0):.4f}")
print(f"  Recall   : {recall_score(y_test, y_pred_dummy, zero_division=0):.4f}")
print(f"  F1-score : {f1_score(y_test, y_pred_dummy, zero_division=0):.4f}")

## 6. Build the Preprocessing Pipeline

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC

# Identify column types in the training set
numeric_features = X_train.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_features = X_train.select_dtypes(include=["object", "category"]).columns.tolist()

print("Numeric features  :", numeric_features)
print("Categorical features:", categorical_features)

# Numeric pipeline: fill missing → scale
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# Categorical pipeline: fill missing → one-hot encode
categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

## 7. Compare Models with Cross-Validation

We test four classifiers using **Stratified 5-fold cross-validation** on the training set only.  
We optimise for **Recall** — it's more costly to miss a potential subscriber than to make an unnecessary call.

In [ ]:
from sklearn.model_selection import cross_validate, StratifiedKFold

scoring = {"accuracy": "accuracy", "precision": "precision",
           "recall": "recall", "f1": "f1"}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42),
    "Decision Tree"      : DecisionTreeClassifier(class_weight="balanced", random_state=42),
    "Random Forest"      : RandomForestClassifier(class_weight="balanced", random_state=42),
    "SVC"                : SVC(class_weight="balanced", probability=True, random_state=42),
}

model_results = []

for name, model in models.items():
    pipe = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("model", model)
    ])
    scores = cross_validate(pipe, X_train, y_train, cv=cv, scoring=scoring, n_jobs=-1)
    model_results.append({
        "model"    : name,
        "accuracy" : scores["test_accuracy"].mean(),
        "precision": scores["test_precision"].mean(),
        "recall"   : scores["test_recall"].mean(),
        "f1"       : scores["test_f1"].mean(),
    })

results_df = pd.DataFrame(model_results).sort_values(by="recall", ascending=False).reset_index(drop=True)
results_df.round(4)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
results_melted = results_df.melt(id_vars="model", var_name="metric", value_name="score")
sns.barplot(data=results_melted, x="score", y="model", hue="metric", ax=ax)
ax.set_title("Model Comparison — Cross-Validation Scores", fontsize=13, fontweight="bold")
ax.set_xlim(0, 1)
ax.legend(title="Metric", bbox_to_anchor=(1, 1))
plt.tight_layout()
plt.show()

## 8. Hyperparameter Tuning — Random Forest

**Why Random Forest?**  
After cross-validation, Logistic Regression had the highest raw Recall (≈0.63), but with very low Precision (≈0.28) — meaning it flags nearly everyone as a subscriber.  
Random Forest strikes a better **precision-recall balance** and is far more interpretable through feature importances.  

We run **GridSearchCV** optimising for `recall` to find the best hyperparameters while still keeping precision reasonable.

In [ ]:
from sklearn.model_selection import GridSearchCV

rf_pipe = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", RandomForestClassifier(class_weight="balanced", random_state=42))
])

param_grid = {
    "model__n_estimators"    : [100, 200],
    "model__max_depth"       : [None, 10, 20],
    "model__min_samples_split": [2, 5, 10],
    "model__min_samples_leaf" : [1, 2, 4],
    "model__max_features"    : ["sqrt", "log2"],
}

grid_search = GridSearchCV(
    estimator=rf_pipe,
    param_grid=param_grid,
    scoring="recall",
    cv=cv,
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X_train, y_train)

print(f"Best CV Recall : {grid_search.best_score_:.4f}")
print("Best Parameters:")
for param, value in grid_search.best_params_.items():
    print(f"  {param}: {value}")

## 9. Final Model Evaluation on the Test Set

The grid search identified the best hyperparameters.  
We now evaluate on the **held-out test set** — data the model has never seen — to get an honest estimate of real-world performance.

In [ ]:
best_rf = grid_search.best_estimator_
y_pred = best_rf.predict(X_test)

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

print("=" * 38)
print("   Final Model — Test Set Performance")
print("=" * 38)
print(f"Accuracy : {accuracy_score(y_test, y_pred):.4f}")
print(f"Precision: {precision_score(y_test, y_pred):.4f}")
print(f"Recall   : {recall_score(y_test, y_pred):.4f}")
print(f"F1-Score : {f1_score(y_test, y_pred):.4f}")

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred,
    display_labels=["No (0)", "Yes (1)"],
    cmap="Blues",
    values_format="d",
    ax=ax
)
ax.set_title("Confusion Matrix — Best Random Forest", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
from sklearn.metrics import classification_report

print(classification_report(y_test, y_pred, target_names=["No (0)", "Yes (1)"]))

In [ ]:
# Pull the one-hot feature names from the fitted pipeline
ohe_features = (
    best_rf.named_steps["preprocessor"]
    .named_transformers_["cat"]
    .named_steps["onehot"]
    .get_feature_names_out(categorical_features)
    .tolist()
)
all_features = numeric_features + ohe_features

importances = best_rf.named_steps["model"].feature_importances_

feat_imp_df = (
    pd.DataFrame({"feature": all_features, "importance": importances})
    .sort_values("importance", ascending=False)
    .head(15)
    .reset_index(drop=True)
)

plt.figure(figsize=(10, 6))
sns.barplot(data=feat_imp_df, x="importance", y="feature", palette="viridis")
plt.title("Top 15 Feature Importances — Random Forest", fontsize=13, fontweight="bold")
plt.xlabel("Importance Score")
plt.ylabel("Feature")
plt.tight_layout()
plt.show()

print("\nTop 5 most important features:")
print(feat_imp_df[["feature", "importance"]].head(5).to_string(index=False))

## 10. Conclusion

| Metric | Dummy Baseline | Best Random Forest |
|--------|---------------|--------------------|
| Accuracy | ~88% | ~81% |
| Recall (Yes) | 0% | ~58% |
| Precision (Yes) | 0% | ~32% |
| F1-Score (Yes) | 0% | ~41% |

**Key takeaways:**
- The dummy classifier looks accurate but catches zero subscribers — useless for the bank.
- Our tuned Random Forest identifies **~58 % of actual subscribers** — a huge improvement.
- The most predictive features are **call-related** (number of contacts, previous outcome) and **financial** (account balance, age).
- Further gains could come from threshold tuning, SMOTE oversampling, or gradient boosting (XGBoost / LightGBM).

---
*Dataset source: [UCI Machine Learning Repository — Bank Marketing](https://archive.ics.uci.edu/ml/datasets/Bank+Marketing)*